# load all

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import torch
import numpy as np

from models.model_utils import load_checkpoint
from models.model_utils import encode as encode_lora_sapbert

from models.model import LoraSapbert

from functions.utils import load_config
config = load_config() 
device = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
# load vocabs in data
df_concept_data = (pl.read_parquet(config['vocab_data']['path'] + config['vocab_data']['concept_with_label_count'])
                    .drop_nulls()
                    .with_row_index("idx")
)

# load our SNOMED CT ontology with labels
df_hug_snomed = pl.read_parquet(config['HUG_SNOMED_DATA']['path'] + config['HUG_SNOMED_DATA']['hug_snomed_info'], columns=["id", "label", "concept_type"], use_pyarrow=True)

# get official SNOMED CT release with synonyms
df_official = (pl.read_parquet(config['HUG_SNOMED_DATA']['path'] + config['HUG_SNOMED_DATA']['official_release'], columns=["id", "term"], use_pyarrow=True)
               .rename({"term": "label"})
               .with_columns(concept_type = pl.lit("SCT_PRE"))
               )

# get combined SNOMED CT references that we can map
df_map_ref = (pl.concat([df_hug_snomed, df_official])
              .unique()
              .with_columns(pl.col("label").str.replace_all(r"\d+\|", "|").alias("label"))
              .drop_nulls()

              .with_row_index("idx")
              )


In [4]:
# load model
model_emb_ft = LoraSapbert(**config["SAPBERT_PARAMETERS"]).to(device)
_ = load_checkpoint(model_emb_ft, config["EMBEDDING_CONCEPT"]["ft_model"], device="cuda", strict=False)
model_emb_ft.eval()

trainable params: 442,368 || all params: 109,924,608 || trainable%: 0.4024


c:\Users\yy\Desktop\codes_PhD\EHRSHOT_tokenizer\models\model_utils.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


LoraSapbert(
  (model): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=

# 1. embed concepts' labels
- Our SNOMED (both FSN nd synonyms), then clean post to remove symbols
- EHRSHOT data
  

In [5]:
labels_ref = df_map_ref["label"].to_list()
cpt_label_ref_embs = encode_lora_sapbert(model_emb_ft, labels_ref, batch_size=2048, device="cuda", head="q", return_dtype=torch.float32)
np.save(config["EMBEDDING_CONCEPT"]["cpt_snomed_ref_emb"], cpt_label_ref_embs)
df_map_ref.write_parquet(config["EMBEDDING_CONCEPT"]["cpt_snomed_ref_info"])


Encoding:   0%|          | 0/577 [00:00<?, ?it/s]

In [6]:
labels_data = df_concept_data["label"].to_list()
cpt_label_data_embs = encode_lora_sapbert(model_emb_ft, labels_data, batch_size=2048, device="cuda", head="q", return_dtype=torch.float32)
np.save(config["EMBEDDING_CONCEPT"]["cpt_ehrshot_emb"], cpt_label_data_embs)
df_concept_data.write_parquet(config["EMBEDDING_CONCEPT"]["cpt_ehrshot_info"])

Encoding:   0%|          | 0/16 [00:00<?, ?it/s]

In [7]:
df_concept_data.drop_nulls().describe()

statistic,idx,code_type,code_value,label,bool_map_direct,count
str,f64,str,str,str,f64,f64
"""count""",30759.0,"""30759""","""30759""","""30759""",30759.0,30759.0
"""null_count""",0.0,"""0""","""0""","""0""",0.0,0.0
"""mean""",15379.0,null,null,null,0.347638,2006.053058
"""std""",8879.502801,null,null,null,null,33032.787587
"""min""",0.0,"""CMS Place of Service""","""00.02""","""0.1 ML Vancomycin 10 MG/ML Inj…",0.0,1.0
"""25%""",7690.0,null,null,null,null,2.0
"""50%""",15379.0,null,null,null,null,9.0
"""75%""",23069.0,null,null,null,null,63.0
"""max""",30758.0,"""Visit""","""XW13325""","""zoster vaccine, live""",1.0,2.615625e6
